# hunch LoRA Adapter Training

Train a custom adapter for Apple's 3B on-device Foundation Model.

**Requirements:**
- Colab Pro (A100 GPU) or any CUDA machine with 24GB+ VRAM
- Apple's adapter training toolkit zip uploaded to Google Drive
- hunch training files (prepare_data.py, bank db, benchmark prompts)

## 1. Setup

Upload to `My Drive/hunch-training/`:
- `adapter_training_toolkit_v26_0_0.zip` (from developer.apple.com)
- `prepare_data.py`
- `tldr_bank.db` (from bank/)
- `prompts.jsonl` (from benchmark/)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/hunch-training'
WORK_DIR = '/content/hunch-training'

In [ ]:
# Extract toolkit (only on first run)
import os
if not os.path.exists(f'{DRIVE_DIR}/adapter_training_toolkit_v26_0_0'):
    !cd {DRIVE_DIR} && unzip -q adapter_training_toolkit_v26_0_0.zip
    print('Extracted toolkit')
else:
    print('Toolkit already extracted')

In [ ]:
# Copy to local VM (faster I/O during training)
!mkdir -p {WORK_DIR}
!cp -r {DRIVE_DIR}/adapter_training_toolkit_v26_0_0 {WORK_DIR}/
!cp {DRIVE_DIR}/prepare_data.py {WORK_DIR}/

# Set up bank and benchmark paths for prepare_data.py
!mkdir -p {WORK_DIR}/../bank {WORK_DIR}/../benchmark
!cp {DRIVE_DIR}/tldr_bank.db {WORK_DIR}/../bank/
!cp {DRIVE_DIR}/prompts.jsonl {WORK_DIR}/../benchmark/
!ls {WORK_DIR}

In [ ]:
# Install dependencies
!cd {WORK_DIR}/adapter_training_toolkit_v26_0_0 && pip install -r requirements.txt -q

In [ ]:
# Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

## 2. Prepare Training Data

In [ ]:
!cd {WORK_DIR} && python3 prepare_data.py --stats

In [ ]:
!cd {WORK_DIR} && python3 prepare_data.py

## 3. Train Adapter

~30-60 minutes on A100. Reduce batch_size to 2 if OOM.

In [ ]:
!cd {WORK_DIR}/adapter_training_toolkit_v26_0_0 && python3 -m examples.train_adapter \
  --train-data ../train.jsonl \
  --eval-data ../eval.jsonl \
  --epochs 5 \
  --learning-rate 1e-3 \
  --batch-size 4 \
  --precision f16 \
  --activation-checkpointing \
  --checkpoint-dir ../checkpoints/

## 4. Evaluate

In [ ]:
test_prompts = [
    'find files changed in the last hour',
    'show disk usage',
    'generate a random password',
    'kill a process by name',
    'show http headers of a url',
    'record terminal session',
    'find files larger than 100mb',
    'convert image to different format',
    'show all listening ports',
    'find files modified in the last 7 days',
    'find files owned by root',
    'count lines in all python files',
    'show all environment variables',
    'clear the terminal',
    'compare two files',
]

import subprocess
for prompt in test_prompts:
    result = subprocess.run(
        ['python3', '-m', 'examples.generate',
         '--prompt', prompt,
         '--checkpoint', '../checkpoints/adapter-final.pt',
         '--max-tokens', '50',
         '--precision', 'f16'],
        capture_output=True, text=True,
        cwd=f'{WORK_DIR}/adapter_training_toolkit_v26_0_0'
    )
    output = result.stdout.strip().split('\n')[-1] if result.stdout else result.stderr[:100]
    print(f'Q: {prompt:<45} A: {output}')

## 5. Compare: Base vs Adapted

In [ ]:
for prompt in test_prompts:
    # Base model (no adapter)
    base = subprocess.run(
        ['python3', '-m', 'examples.generate',
         '--prompt', prompt, '--max-tokens', '50', '--precision', 'f16'],
        capture_output=True, text=True,
        cwd=f'{WORK_DIR}/adapter_training_toolkit_v26_0_0'
    )
    # Adapted model
    adapted = subprocess.run(
        ['python3', '-m', 'examples.generate',
         '--prompt', prompt,
         '--checkpoint', '../checkpoints/adapter-final.pt',
         '--max-tokens', '50', '--precision', 'f16'],
        capture_output=True, text=True,
        cwd=f'{WORK_DIR}/adapter_training_toolkit_v26_0_0'
    )
    b = base.stdout.strip().split('\n')[-1] if base.stdout else '?'
    a = adapted.stdout.strip().split('\n')[-1] if adapted.stdout else '?'
    print(f'Q: {prompt}')
    print(f'  base:    {b}')
    print(f'  adapted: {a}')
    print()

## 6. Export .fmadapter

In [ ]:
!cd {WORK_DIR}/adapter_training_toolkit_v26_0_0 && python3 -m export.export_fmadapter \
  --adapter-name hunch \
  --checkpoint ../checkpoints/adapter-final.pt \
  --output-dir ../exports/

!ls -lh {WORK_DIR}/exports/

In [ ]:
# Copy results back to Google Drive
!cp -r {WORK_DIR}/exports {DRIVE_DIR}/
!cp -r {WORK_DIR}/checkpoints {DRIVE_DIR}/
print('Saved to Google Drive. Download hunch.fmadapter and test on macOS 26.')